# Notebook 03 — Base Analítica e Engenharia de Features

Este notebook constrói as matrizes de features para desenvolvimento e scoring com a mesma lógica point-in-time. O foco é produzir variáveis disponíveis no momento da solicitação, preservar rastreabilidade e antecipar incompatibilidades de aplicação sem utilizar a base de scoring no aprendizado.

**Escopo**

- definir uma unidade de avaliação comum;
- construir features da operação, cadastro, histórico de crédito e pagamentos;
- controlar leakage, cobertura, variabilidade e compatibilidade de schema;
- salvar as saídas que serão consumidas pelo Notebook 04.

## 1. Configuração e preparação das fontes

In [1]:
from pathlib import Path
import os
import sys

from pyspark.sql import Row, SparkSession, Window
from pyspark.sql import functions as F

# Garante que driver e workers do Spark utilizem o Python do kernel ativo.
PYTHON_EXECUTAVEL = str(Path(sys.executable).resolve())

os.environ["PYSPARK_PYTHON"] = PYTHON_EXECUTAVEL
os.environ["PYSPARK_DRIVER_PYTHON"] = PYTHON_EXECUTAVEL
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"

try:
    spark.stop()
except (NameError, AttributeError):
    pass

spark = (
    SparkSession.builder
    .master(os.getenv("SPARK_MASTER", "local[2]"))
    .appName("credit-risk-case")
    .config("spark.ui.enabled", "false")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.pyspark.python", PYTHON_EXECUTAVEL)
    .config("spark.pyspark.driver.python", PYTHON_EXECUTAVEL)
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.adaptive.enabled", "true")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
# Ajuste manual opcional. Na ausência de valor, são utilizadas variáveis de ambiente,
# um Volume padrão do Databricks, quando disponível, ou pastas locais relativas.
CAMINHO_DADOS_MANUAL = None
CAMINHO_SAIDA_MANUAL = None


def resolver_caminho(caminho_manual, variavel_ambiente, caminho_databricks, caminho_local):
    if caminho_manual:
        return str(caminho_manual)
    if os.getenv(variavel_ambiente):
        return os.environ[variavel_ambiente]
    if caminho_databricks and Path(caminho_databricks).exists():
        return caminho_databricks
    return caminho_local


def juntar_caminho(diretorio, arquivo):
    return f"{str(diretorio).rstrip('/')}/{arquivo}"


DATA_PATH = resolver_caminho(
    CAMINHO_DADOS_MANUAL,
    "CREDIT_RISK_DATA_PATH",
    "/Volumes/workspace/default/credit_risk_data",
    "data"
)

OUTPUT_PATH = resolver_caminho(
    CAMINHO_SAIDA_MANUAL,
    "CREDIT_RISK_OUTPUT_PATH",
    "/Volumes/workspace/default/case_tecnico_ds",
    "outputs"
)

if "://" not in OUTPUT_PATH:
    Path(OUTPUT_PATH).mkdir(parents=True, exist_ok=True)


def visualizar(df, n=20, truncate=False):
    if n <= 0:
        raise ValueError("O parâmetro 'n' deve ser maior que zero.")
    df.show(n=n, truncate=truncate)


print(f"Python: {sys.version.split()[0]} | Spark: {spark.version}")
print(f"Dados: {DATA_PATH}")
print(f"Saídas: {OUTPUT_PATH}")

Python: 3.11.7 | Spark: 4.1.2
Dados: data
Saídas: outputs


In [2]:
base_cadastral_raw = spark.read.parquet(juntar_caminho(DATA_PATH, "base_cadastral.parquet"))
base_submissao_raw = spark.read.parquet(juntar_caminho(DATA_PATH, "base_submissao.parquet"))
historico_emprestimos_raw = spark.read.parquet(juntar_caminho(DATA_PATH, "historico_emprestimos.parquet"))
historico_parcelas_raw = spark.read.parquet(juntar_caminho(DATA_PATH, "historico_parcelas.parquet"))

def converter_colunas_data(dataframe, colunas):
    resultado = dataframe
    for coluna in colunas:
        if coluna in resultado.columns:
            resultado = resultado.withColumn(coluna, F.to_date(F.col(coluna), "yyyy-MM-dd"))
    return resultado

base_cadastral = converter_colunas_data(base_cadastral_raw, ["data_nascimento"])
base_submissao = converter_colunas_data(base_submissao_raw, ["data_solicitacao"])
historico_emprestimos = converter_colunas_data(
    historico_emprestimos_raw,
    [
        "data_decisao", "data_liberacao", "data_primeiro_vencimento",
        "data_ultimo_vencimento_original", "data_ultimo_vencimento", "data_encerramento"
    ]
)
historico_parcelas = converter_colunas_data(
    historico_parcelas_raw,
    ["data_prevista_pagamento", "data_real_pagamento"]
)

In [3]:
bases_carregadas = {
    "base_cadastral": base_cadastral,
    "base_submissao": base_submissao,
    "historico_emprestimos": historico_emprestimos,
    "historico_parcelas": historico_parcelas
}

validacao_carregamento = spark.createDataFrame([
    Row(tabela=nome, qtd_registros=df.count(), qtd_colunas=len(df.columns))
    for nome, df in bases_carregadas.items()
])
visualizar(validacao_carregamento)

+---------------------+-------------+-----------+
|tabela               |qtd_registros|qtd_colunas|
+---------------------+-------------+-----------+
|base_cadastral       |40000        |16         |
|base_submissao       |40000        |8          |
|historico_emprestimos|186890       |37         |
|historico_parcelas   |1390978      |8          |
+---------------------+-------------+-----------+



## 2. Unidades de avaliação e corte temporal

No desenvolvimento, cada avaliação corresponde a um contrato aprovado com parcelas observáveis. No scoring, cada avaliação corresponde à solicitação recente de um cliente. A `data_corte_features` é o menor valor entre a data da avaliação e o encerramento conhecido do histórico, impedindo acesso a eventos futuros.

In [4]:
contratos_com_historico_parcelas = (
    historico_parcelas
    .select("id_contrato", "id_cliente")
    .distinct()
)

contratos_elegiveis_desenvolvimento = (
    historico_emprestimos
    .filter(F.col("status_contrato") == "Approved")
    .join(
        contratos_com_historico_parcelas,
        on=["id_contrato", "id_cliente"],
        how="left_semi"
    )
)

base_avaliacao_desenvolvimento = (
    contratos_elegiveis_desenvolvimento
    .select(
        F.concat(F.lit("DEV_"), F.col("id_contrato").cast("string")).alias("id_avaliacao"),
        F.lit("DESENVOLVIMENTO").alias("tipo_base"),
        "id_contrato", "id_cliente",
        F.col("data_decisao").alias("data_avaliacao"),
        "tipo_contrato", "dia_semana_solicitacao", "hora_solicitacao",
        F.col("valor_solicitado").alias("valor_credito_solicitado"),
        "valor_bem", "valor_parcela"
    )
)

base_avaliacao_scoring = (
    base_submissao
    .select(
        F.concat(F.lit("SCORE_"), F.col("id_cliente").cast("string")).alias("id_avaliacao"),
        F.lit("SCORING").alias("tipo_base"),
        F.lit(None).cast("long").alias("id_contrato"),
        "id_cliente",
        F.col("data_solicitacao").alias("data_avaliacao"),
        "tipo_contrato", "dia_semana_solicitacao", "hora_solicitacao",
        F.col("valor_credito").alias("valor_credito_solicitado"),
        "valor_bem", "valor_parcela"
    )
)

DATA_CORTE_HISTORICO = "2025-02-22"

bases_avaliacao = (
    base_avaliacao_desenvolvimento
    .unionByName(base_avaliacao_scoring)
    .withColumn(
        "data_corte_features",
        F.least(F.col("data_avaliacao"), F.lit(DATA_CORTE_HISTORICO).cast("date"))
    )
)

In [5]:
validacao_bases_avaliacao = (
    bases_avaliacao
    .groupBy("tipo_base")
    .agg(
        F.count("*").alias("qtd_avaliacoes"),
        F.countDistinct("id_avaliacao").alias("qtd_avaliacoes_distintas"),
        F.countDistinct("id_cliente").alias("qtd_clientes"),
        F.min("data_avaliacao").alias("menor_data_avaliacao"),
        F.max("data_avaliacao").alias("maior_data_avaliacao"),
        F.min("data_corte_features").alias("menor_data_corte"),
        F.max("data_corte_features").alias("maior_data_corte")
    )
    .withColumn("qtd_duplicadas", F.col("qtd_avaliacoes") - F.col("qtd_avaliacoes_distintas"))
    .orderBy("tipo_base")
)
visualizar(validacao_bases_avaliacao)

validacao_cobertura_cadastral = (
    bases_avaliacao.select("tipo_base", "id_avaliacao", "id_cliente")
    .join(base_cadastral.select("id_cliente").distinct().withColumn("flag_cadastro", F.lit(1)), "id_cliente", "left")
    .groupBy("tipo_base")
    .agg(
        F.count("*").alias("qtd_avaliacoes"),
        F.sum(F.col("flag_cadastro").isNull().cast("int")).alias("qtd_sem_cadastro")
    )
)
visualizar(validacao_cobertura_cadastral)

+---------------+--------------+------------------------+------------+--------------------+--------------------+----------------+----------------+--------------+
|tipo_base      |qtd_avaliacoes|qtd_avaliacoes_distintas|qtd_clientes|menor_data_avaliacao|maior_data_avaliacao|menor_data_corte|maior_data_corte|qtd_duplicadas|
+---------------+--------------+------------------------+------------+--------------------+--------------------+----------------+----------------+--------------+
|DESENVOLVIMENTO|107419        |107419                  |37739       |2017-02-04          |2025-02-14          |2017-02-04      |2025-02-14      |0             |
|SCORING        |40000         |40000                   |40000       |2025-02-04          |2025-02-24          |2025-02-04      |2025-02-22      |0             |
+---------------+--------------+------------------------+------------+--------------------+--------------------+----------------+----------------+--------------+

+---------------+----------

**Decisão temporal**

O histórico associado a cada avaliação deve satisfazer `data_evento < data_corte_features`. A base de scoring é preparada agora somente para validar a execução do mesmo pipeline; ela não participa da seleção de variáveis, imputação, ajuste de hiperparâmetros ou treinamento.

## 3. Features da operação

Valores não positivos são tratados como indisponíveis e preservados por flags. O sufixo `_valido` descreve exatamente a transformação: somente valores estritamente positivos permanecem. Informações sobre o valor do bem são mantidas para auditoria, mas não serão usadas diretamente porque a razão crédito/bem é constante no desenvolvimento e variável no scoring.

In [6]:
bases_avaliacao_operacao = (
    bases_avaliacao
    .withColumn(
        "flag_valor_credito_solicitado_indisponivel",
        (F.col("valor_credito_solicitado").isNull() | (F.col("valor_credito_solicitado") <= 0)).cast("int")
    )
    .withColumn(
        "flag_valor_bem_indisponivel",
        (F.col("valor_bem").isNull() | (F.col("valor_bem") <= 0)).cast("int")
    )
    .withColumn(
        "flag_valor_parcela_indisponivel",
        (F.col("valor_parcela").isNull() | (F.col("valor_parcela") <= 0)).cast("int")
    )
    .withColumn(
        "valor_credito_solicitado_valido",
        F.when(F.col("valor_credito_solicitado") > 0, F.col("valor_credito_solicitado"))
    )
    .withColumn("valor_bem_valido", F.when(F.col("valor_bem") > 0, F.col("valor_bem")))
    .withColumn("valor_parcela_valido", F.when(F.col("valor_parcela") > 0, F.col("valor_parcela")))
)

features_operacao = (
    bases_avaliacao_operacao
    .withColumn(
        "razao_credito_bem",
        F.when(
            F.col("valor_credito_solicitado_valido").isNotNull() & F.col("valor_bem_valido").isNotNull(),
            F.col("valor_credito_solicitado_valido") / F.col("valor_bem_valido")
        )
    )
    .withColumn(
        "razao_parcela_credito",
        F.when(
            F.col("valor_parcela_valido").isNotNull() & F.col("valor_credito_solicitado_valido").isNotNull(),
            F.col("valor_parcela_valido") / F.col("valor_credito_solicitado_valido")
        )
    )
    .withColumn(
        "log_valor_credito_solicitado",
        F.when(F.col("valor_credito_solicitado_valido").isNotNull(), F.log1p("valor_credito_solicitado_valido"))
    )
    .withColumn("log_valor_bem", F.when(F.col("valor_bem_valido").isNotNull(), F.log1p("valor_bem_valido")))
    .withColumn(
        "log_valor_parcela",
        F.when(F.col("valor_parcela_valido").isNotNull(), F.log1p("valor_parcela_valido"))
    )
    .withColumn(
        "flag_fim_semana",
        F.when(F.col("dia_semana_solicitacao").isNull(), F.lit(None).cast("int"))
        .when(F.col("dia_semana_solicitacao").isin("SATURDAY", "SUNDAY"), 1)
        .otherwise(0)
    )
    .withColumn(
        "periodo_solicitacao",
        F.when(F.col("hora_solicitacao").isNull(), "DESCONHECIDO")
        .when(F.col("hora_solicitacao").between(0, 5), "MADRUGADA")
        .when(F.col("hora_solicitacao").between(6, 11), "MANHA")
        .when(F.col("hora_solicitacao").between(12, 17), "TARDE")
        .otherwise("NOITE")
    )
    .withColumn(
        "flag_horario_comercial",
        F.when(F.col("hora_solicitacao").isNull(), F.lit(None).cast("int"))
        .when(F.col("hora_solicitacao").between(8, 18), 1)
        .otherwise(0)
    )
)

FEATURES_OPERACAO_CANDIDATAS = [
    "tipo_contrato", "dia_semana_solicitacao", "hora_solicitacao",
    "valor_credito_solicitado_valido", "valor_parcela_valido",
    "log_valor_credito_solicitado", "log_valor_parcela", "razao_parcela_credito",
    "flag_valor_credito_solicitado_indisponivel", "flag_valor_parcela_indisponivel",
    "flag_fim_semana", "periodo_solicitacao", "flag_horario_comercial"
]

COLUNAS_OPERACAO_SOMENTE_AUDITORIA = [
    "valor_credito_solicitado", "valor_parcela", "valor_bem",
    "valor_bem_valido", "log_valor_bem", "razao_credito_bem",
    "flag_valor_bem_indisponivel"
]

In [7]:
validacao_operacao = (
    features_operacao
    .groupBy("tipo_base")
    .agg(
        F.count("*").alias("qtd_avaliacoes"),
        F.sum("flag_valor_credito_solicitado_indisponivel").alias("qtd_credito_indisponivel"),
        F.sum("flag_valor_parcela_indisponivel").alias("qtd_parcela_indisponivel"),
        F.sum("flag_valor_bem_indisponivel").alias("qtd_bem_indisponivel"),
        F.min("razao_credito_bem").alias("min_razao_credito_bem"),
        F.max("razao_credito_bem").alias("max_razao_credito_bem"),
        F.min("razao_parcela_credito").alias("min_razao_parcela_credito"),
        F.max("razao_parcela_credito").alias("max_razao_parcela_credito")
    )
    .orderBy("tipo_base")
)
visualizar(validacao_operacao)

+---------------+--------------+------------------------+------------------------+--------------------+---------------------+---------------------+-------------------------+-------------------------+
|tipo_base      |qtd_avaliacoes|qtd_credito_indisponivel|qtd_parcela_indisponivel|qtd_bem_indisponivel|min_razao_credito_bem|max_razao_credito_bem|min_razao_parcela_credito|max_razao_parcela_credito|
+---------------+--------------+------------------------+------------------------+--------------------+---------------------+---------------------+-------------------------+-------------------------+
|DESENVOLVIMENTO|107419        |3849                    |136                     |3849                |1.0                  |1.0                  |0.005                    |0.4080478671442527       |
|SCORING        |40000         |0                       |4                       |24                  |0.3                  |2.6666666666666665   |0.02207258064516129      |0.12442776735459662      |


**Decisão**

`razao_credito_bem` não será candidata: no desenvolvimento, os valores disponíveis resultam em razão igual a 1, enquanto o scoring apresenta variação. O modelo não teria base histórica para aprender esse comportamento. A exclusão reduz risco de aplicação fora do suporte observado.

## 4. Features cadastrais

A idade é calculada na data da avaliação, não na data de execução. Renda por membro é uma razão direta, por isso o nome não utiliza o sufixo genérico `proxy`. `sexo` será preservado apenas para auditoria de equidade e não será oferecido diretamente aos modelos. Variáveis territoriais permanecem candidatas, condicionadas à avaliação de estabilidade e impacto.

In [8]:
COLUNAS_CADASTRAIS = [
    "id_cliente", "data_nascimento", "sexo", "qtd_filhos",
    "qtd_membros_familia", "renda_anual", "tipo_renda", "ocupacao",
    "tipo_organizacao", "nivel_educacao", "estado_civil", "tipo_moradia",
    "possui_carro", "possui_imovel", "nota_regiao_cliente",
    "nota_regiao_cliente_cidade"
]

cadastro_para_modelagem = base_cadastral.select(*COLUNAS_CADASTRAIS)

condicao_renda_por_membro = (
    (F.col("renda_anual") > 0)
    & (F.col("qtd_membros_familia") > 0)
)

features_cadastrais = (
    features_operacao
    .join(cadastro_para_modelagem, on="id_cliente", how="left")
    .withColumn(
        "idade_avaliacao",
        F.floor(F.months_between("data_avaliacao", "data_nascimento") / 12)
    )
    .withColumn(
        "log_renda_anual",
        F.when(F.col("renda_anual") > 0, F.log1p("renda_anual"))
    )
    .withColumn(
        "renda_anual_por_membro",
        F.when(
            condicao_renda_por_membro,
            F.col("renda_anual") / F.col("qtd_membros_familia")
        )
    )
    .withColumn(
        "flag_possui_filhos",
        F.when(F.col("qtd_filhos") > 0, 1).otherwise(0)
    )
)

In [9]:
# Correção semântica: ausência de quantidade de filhos não equivale a ausência de filhos.
features_cadastrais = features_cadastrais.withColumn(
    "flag_possui_filhos",
    F.when(F.col("qtd_filhos").isNull(), F.lit(None).cast("int"))
    .when(F.col("qtd_filhos") > 0, 1)
    .otherwise(0)
)

FEATURES_CADASTRAIS_NUMERICAS_CANDIDATAS = [
    "idade_avaliacao", "qtd_filhos", "qtd_membros_familia", "renda_anual",
    "log_renda_anual", "renda_anual_por_membro", "flag_possui_filhos",
    "nota_regiao_cliente", "nota_regiao_cliente_cidade"
]

FEATURES_CADASTRAIS_CATEGORICAS_CANDIDATAS = [
    "sexo", "tipo_renda", "ocupacao", "tipo_organizacao", "nivel_educacao",
    "estado_civil", "tipo_moradia", "possui_carro", "possui_imovel"
]

COLUNAS_CADASTRAIS_SOMENTE_AUDITORIA = ["sexo"]
COLUNAS_CADASTRAIS_NAO_MODELAVEIS_DIRETAS = ["id_cliente", "data_nascimento"]

In [10]:
validacao_cadastral = (
    features_cadastrais
    .groupBy("tipo_base")
    .agg(
        F.count("*").alias("qtd_avaliacoes"),
        F.countDistinct("id_avaliacao").alias("qtd_avaliacoes_distintas"),
        F.min("idade_avaliacao").alias("min_idade"),
        F.percentile_approx("idade_avaliacao", 0.5).alias("mediana_idade"),
        F.max("idade_avaliacao").alias("max_idade"),
        F.sum(F.col("ocupacao").isNull().cast("int")).alias("qtd_ocupacao_nula")
    )
    .withColumn("qtd_duplicadas", F.col("qtd_avaliacoes") - F.col("qtd_avaliacoes_distintas"))
    .orderBy("tipo_base")
)
visualizar(validacao_cadastral)

+---------------+--------------+------------------------+---------+-------------+---------+-----------------+--------------+
|tipo_base      |qtd_avaliacoes|qtd_avaliacoes_distintas|min_idade|mediana_idade|max_idade|qtd_ocupacao_nula|qtd_duplicadas|
+---------------+--------------+------------------------+---------+-------------+---------+-----------------+--------------+
|DESENVOLVIMENTO|107419        |107419                  |18       |40           |68       |35163            |0             |
|SCORING        |40000         |40000                   |20       |43           |68       |12676            |0             |
+---------------+--------------+------------------------+---------+-------------+---------+-----------------+--------------+



## 5. Features históricas de crédito

O relacionamento point-in-time considera apenas contratos com decisão anterior ao corte da avaliação. Datas de liberação e encerramento também são comparadas ao corte antes de gerar exposição ou status de atividade.

In [11]:
historico_credito_preparado = (
    historico_emprestimos
    .select(
        "id_contrato",
        "id_cliente",
        "data_decisao",
        "data_liberacao",
        "data_encerramento",
        "tipo_contrato",
        "status_contrato",
        "valor_solicitado",
        "valor_credito",
        "valor_bem",
        "valor_parcela",
        "valor_entrada",
        "percentual_entrada",
        "qtd_parcelas_planejadas",
        "taxa_juros_padrao",
        "taxa_juros_promocional",
        "tipo_pagamento",
        "finalidade_emprestimo",
        "tipo_cliente",
        "tipo_portfolio",
        "tipo_produto",
        "categoria_bem",
        "canal_venda"
    )
    .filter(
        F.col("data_decisao").isNotNull()
    )
)

In [12]:
COLUNAS_AVALIACAO_HISTORICO = [
    "id_avaliacao", "tipo_base", "id_cliente", "id_contrato_avaliado",
    "data_avaliacao", "data_corte_features"
]

MAPEAMENTO_HISTORICO = [
    ("id_contrato", "id_contrato_historico"),
    ("data_decisao", "data_decisao_historica"),
    ("data_liberacao", "data_liberacao_historica"),
    ("data_encerramento", "data_encerramento_historica"),
    ("tipo_contrato", "tipo_contrato_historico"),
    ("status_contrato", "status_contrato_historico"),
    ("valor_solicitado", "valor_solicitado_historico"),
    ("valor_credito", "valor_credito_liberado_historico"),
    ("valor_bem", "valor_bem_historico"),
    ("valor_parcela", "valor_parcela_historica"),
    ("valor_entrada", "valor_entrada_historico"),
    ("percentual_entrada", "percentual_entrada_historico"),
    ("qtd_parcelas_planejadas", "qtd_parcelas_planejadas_historico"),
    ("taxa_juros_padrao", "taxa_juros_padrao_historica"),
    ("taxa_juros_promocional", "taxa_juros_promocional_historica"),
    ("tipo_pagamento", "tipo_pagamento_historico"),
    ("finalidade_emprestimo", "finalidade_emprestimo_historica"),
    ("tipo_cliente", "tipo_cliente_historico"),
    ("tipo_portfolio", "tipo_portfolio_historico"),
    ("tipo_produto", "tipo_produto_historico"),
    ("categoria_bem", "categoria_bem_historica"),
    ("canal_venda", "canal_venda_historico")
]

avaliacoes_para_historico = (
    features_cadastrais
    .select(
        "id_avaliacao", "tipo_base", "id_cliente",
        F.col("id_contrato").alias("id_contrato_avaliado"),
        "data_avaliacao", "data_corte_features"
    )
)

condicao_historico_anterior = (
    (F.col("avaliacao.id_cliente") == F.col("historico.id_cliente"))
    & (F.col("historico.data_decisao") < F.col("avaliacao.data_corte_features"))
)

historico_credito_pit = (
    avaliacoes_para_historico.alias("avaliacao")
    .join(
        historico_credito_preparado.alias("historico"),
        on=condicao_historico_anterior,
        how="left"
    )
    .select(
        *[F.col(f"avaliacao.{coluna}") for coluna in COLUNAS_AVALIACAO_HISTORICO],
        *[
            F.col(f"historico.{origem}").alias(destino)
            for origem, destino in MAPEAMENTO_HISTORICO
        ]
    )
    .withColumn(
        "dias_desde_decisao_historica",
        F.datediff("data_corte_features", "data_decisao_historica")
    )
)

In [13]:
MAPA_STATUS = {
    "Approved": "aprovadas",
    "Refused": "recusadas",
    "Canceled": "canceladas",
    "Unused offer": "ofertas_nao_utilizadas"
}

JANELAS_RECENCIA = [30, 90, 180, 365]

status_historico = F.col("status_contrato_historico")
dias_desde_decisao = F.col("dias_desde_decisao_historica")

agregacoes_status = [
    F.sum(F.when(status_historico == status, 1).otherwise(0))
        .alias(f"qtd_{sufixo}_anteriores")
    for status, sufixo in MAPA_STATUS.items()
]

agregacoes_recencia = [
    F.sum(F.when(dias_desde_decisao.between(1, dias), 1).otherwise(0))
        .alias(f"qtd_solicitacoes_ultimos_{dias}d")
    for dias in JANELAS_RECENCIA
]

features_historicas_credito_status = (
    historico_credito_pit
    .groupBy("id_avaliacao")
    .agg(
        F.first("tipo_base", ignorenulls=True).alias("tipo_base"),
        F.count("id_contrato_historico").alias("qtd_solicitacoes_anteriores"),
        *agregacoes_status,
        *agregacoes_recencia,
        F.min(dias_desde_decisao).alias("dias_desde_ultima_solicitacao"),
        F.min(F.when(status_historico == "Approved", dias_desde_decisao))
            .alias("dias_desde_ultima_aprovacao"),
        F.min(F.when(status_historico == "Refused", dias_desde_decisao))
            .alias("dias_desde_ultima_recusa")
    )
)

total_solicitacoes = F.col("qtd_solicitacoes_anteriores")
total_status = sum(
    F.col(f"qtd_{sufixo}_anteriores")
    for sufixo in MAPA_STATUS.values()
)

features_historicas_credito_status = (
    features_historicas_credito_status
    .select(
        "*",
        F.when(total_solicitacoes == 0, 1).otherwise(0)
            .alias("flag_sem_historico_credito"),
        F.when(total_solicitacoes > 0, F.col("qtd_aprovadas_anteriores") / total_solicitacoes * 100)
            .alias("perc_aprovadas_anteriores"),
        F.when(total_solicitacoes > 0, F.col("qtd_recusadas_anteriores") / total_solicitacoes * 100)
            .alias("perc_recusadas_anteriores"),
        F.when(total_solicitacoes > 0, F.col("qtd_canceladas_anteriores") / total_solicitacoes * 100)
            .alias("perc_canceladas_anteriores"),
        total_status.alias("qtd_status_classificados"),
        F.when(total_solicitacoes != total_status, 1).otherwise(0)
            .alias("flag_inconsistencia_soma_status")
    )
)

In [14]:
def contar_quando(condicao, nome):
    return F.sum(F.when(condicao, 1).otherwise(0)).alias(nome)


valor_liberado = F.col("valor_credito_liberado_historico")
valor_solicitado = F.col("valor_solicitado_historico")
valor_parcela = F.col("valor_parcela_historica")
qtd_parcelas = F.col("qtd_parcelas_planejadas_historico")
data_corte = F.col("data_corte_features")

condicao_aprovado_liberado_ate_corte = (
    (F.col("status_contrato_historico") == "Approved")
    & F.col("data_liberacao_historica").isNotNull()
    & (F.col("data_liberacao_historica") <= data_corte)
)

condicao_credito_liberado_valido = (
    condicao_aprovado_liberado_ate_corte
    & valor_liberado.isNotNull()
    & (valor_liberado > 0)
)

condicao_comparacao_solicitado_liberado = (
    condicao_credito_liberado_valido
    & valor_solicitado.isNotNull()
    & (valor_solicitado > 0)
)

condicao_contrato_ativo_no_corte = (
    condicao_credito_liberado_valido
    & (
        F.col("data_encerramento_historica").isNull()
        | (F.col("data_encerramento_historica") > data_corte)
    )
)

condicao_valor_parcela_valido = (
    condicao_aprovado_liberado_ate_corte
    & valor_parcela.isNotNull()
    & (valor_parcela > 0)
)

condicao_qtd_parcelas_valida = (
    condicao_aprovado_liberado_ate_corte
    & qtd_parcelas.isNotNull()
    & (qtd_parcelas > 0)
)

features_historicas_credito_valores = (
    historico_credito_pit
    .groupBy("id_avaliacao")
    .agg(
        F.first("tipo_base", ignorenulls=True).alias("tipo_base"),

        contar_quando(
            condicao_aprovado_liberado_ate_corte,
            "qtd_contratos_liberados_anteriores"
        ),
        contar_quando(
            condicao_credito_liberado_valido,
            "qtd_contratos_com_valor_liberado_valido"
        ),
        contar_quando(
            condicao_contrato_ativo_no_corte,
            "qtd_contratos_ativos_no_corte"
        ),

        F.sum(F.when(condicao_credito_liberado_valido, valor_liberado))
            .alias("soma_valor_credito_liberado_anterior"),
        F.avg(F.when(condicao_credito_liberado_valido, valor_liberado))
            .alias("media_valor_credito_liberado_anterior"),
        F.max(F.when(condicao_credito_liberado_valido, valor_liberado))
            .alias("max_valor_credito_liberado_anterior"),
        F.sum(F.when(condicao_contrato_ativo_no_corte, valor_liberado))
            .alias("soma_valor_liberado_contratos_ativos_no_corte"),

        F.avg(F.when(condicao_valor_parcela_valido, valor_parcela))
            .alias("media_valor_parcela_anterior"),
        F.avg(F.when(condicao_qtd_parcelas_valida, qtd_parcelas))
            .alias("media_qtd_parcelas_planejadas_anterior"),

        contar_quando(
            condicao_comparacao_solicitado_liberado,
            "qtd_contratos_com_comparacao_valores"
        ),
        contar_quando(
            condicao_comparacao_solicitado_liberado
            & (valor_liberado < valor_solicitado),
            "qtd_credito_liberado_menor_solicitado"
        ),

        F.avg(
            F.when(
                condicao_comparacao_solicitado_liberado,
                valor_liberado / valor_solicitado
            )
        ).alias("media_razao_liberado_solicitado"),

        contar_quando(
            condicao_aprovado_liberado_ate_corte
            & (valor_liberado.isNull() | (valor_liberado <= 0)),
            "qtd_contratos_liberados_sem_valor_valido"
        ),
        contar_quando(
            condicao_comparacao_solicitado_liberado
            & (valor_liberado > valor_solicitado),
            "qtd_contratos_liberado_maior_solicitado"
        )
    )
    .withColumn(
        "flag_sem_credito_liberado_anterior",
        F.when(F.col("qtd_contratos_liberados_anteriores") == 0, 1).otherwise(0)
    )
    .withColumn(
        "perc_credito_liberado_menor_solicitado",
        F.when(
            F.col("qtd_contratos_com_comparacao_valores") > 0,
            F.col("qtd_credito_liberado_menor_solicitado")
            / F.col("qtd_contratos_com_comparacao_valores") * 100
        )
    )
    .withColumn(
        "flag_historico_liberado_sem_valor_valido",
        F.when(F.col("qtd_contratos_liberados_sem_valor_valido") > 0, 1).otherwise(0)
    )
    .withColumn(
        "flag_historico_liberado_maior_solicitado",
        F.when(F.col("qtd_contratos_liberado_maior_solicitado") > 0, 1).otherwise(0)
    )
)

In [15]:
validacao_historico_credito = (
    historico_credito_pit
    .groupBy("tipo_base")
    .agg(
        F.countDistinct("id_avaliacao").alias("qtd_avaliacoes_com_linha_pit"),
        F.countDistinct(F.when(F.col("id_contrato_historico").isNotNull(), F.col("id_avaliacao"))).alias("qtd_com_historico"),
        F.countDistinct("id_contrato_historico").alias("qtd_contratos_historicos")
    )
)

validacao_features_credito = (
    features_historicas_credito_status
    .join(
        features_historicas_credito_valores.drop("tipo_base"),
        "id_avaliacao",
        "left"
    )
    .groupBy("tipo_base")
    .agg(
        F.count("*").alias("qtd_avaliacoes"),
        F.round(F.avg("flag_sem_historico_credito") * 100, 4).alias("perc_sem_historico_credito"),
        F.percentile_approx("qtd_solicitacoes_anteriores", 0.5).alias("mediana_solicitacoes_anteriores"),
        F.round(F.avg("flag_sem_credito_liberado_anterior") * 100, 4).alias("perc_sem_credito_liberado")
    )
)
visualizar(validacao_historico_credito.orderBy("tipo_base"))
visualizar(validacao_features_credito.orderBy("tipo_base"))

+---------------+----------------------------+-----------------+------------------------+
|tipo_base      |qtd_avaliacoes_com_linha_pit|qtd_com_historico|qtd_contratos_historicos|
+---------------+----------------------------+-----------------+------------------------+
|DESENVOLVIMENTO|107419                      |70367            |98054                   |
|SCORING        |40000                       |37952            |186884                  |
+---------------+----------------------------+-----------------+------------------------+

+---------------+--------------+--------------------------+-------------------------------+-------------------------+
|tipo_base      |qtd_avaliacoes|perc_sem_historico_credito|mediana_solicitacoes_anteriores|perc_sem_credito_liberado|
+---------------+--------------+--------------------------+-------------------------------+-------------------------+
|DESENVOLVIMENTO|107419        |34.493                    |1                              |93.0841       

**Decisões**

- Recências permanecem nulas quando o evento nunca ocorreu; zero significaria evento no próprio corte.
- Valores históricos permanecem nulos quando não há operação liberada válida; flags distinguem ausência estrutural.
- Variáveis de comparação entre solicitado e liberado serão excluídas mais adiante por cobertura insuficiente no desenvolvimento.
- A diferença de profundidade histórica entre desenvolvimento e scoring será monitorada como risco de estabilidade, não tratada como erro de construção.

## 6. Features históricas de pagamento

A consolidação abaixo reproduz a lógica financeira do Notebook 02 no corte específico de cada avaliação. Pagamentos posteriores ao corte são descartados. A quitação depende do pagamento acumulado, evitando considerar uma parcela parcialmente paga como liquidada.

In [16]:
TOLERANCIA_PAGAMENTO = 0.01
LIMITE_SALDO_RESIDUAL_ABSOLUTO = 5.00
LIMITE_SALDO_RESIDUAL_PERCENTUAL = 0.001

CHAVES_PARCELA_AVALIACAO = [
    "id_avaliacao", "tipo_base", "id_cliente", "id_contrato_historico",
    "data_corte_features", "numero_parcela", "data_prevista_pagamento"
]

eventos_pagamento_historicos = (
    historico_credito_pit.alias("h")
    .join(
        historico_parcelas.alias("p"),
        (F.col("h.id_cliente") == F.col("p.id_cliente")) &
        (F.col("h.id_contrato_historico") == F.col("p.id_contrato")),
        "inner"
    )
    .filter(
        F.col("p.data_prevista_pagamento").isNotNull() &
        (F.col("p.data_prevista_pagamento") < F.col("h.data_corte_features"))
    )
    .select(
        F.col("h.id_avaliacao").alias("id_avaliacao"),
        F.col("h.tipo_base").alias("tipo_base"),
        F.col("h.id_cliente").alias("id_cliente"),
        F.col("h.id_contrato_historico").alias("id_contrato_historico"),
        F.col("h.data_corte_features").alias("data_corte_features"),
        F.col("p.numero_parcela").alias("numero_parcela"),
        F.col("p.versao_parcela").alias("versao_parcela"),
        F.col("p.data_prevista_pagamento").alias("data_prevista_pagamento"),
        F.col("p.data_real_pagamento").alias("data_real_pagamento"),
        F.col("p.valor_previsto_parcela").cast("double").alias("valor_previsto_parcela"),
        F.col("p.valor_pago_parcela").cast("double").alias("valor_pago_parcela")
    )
)

In [17]:
valores_previstos_por_versao = (
    eventos_pagamento_historicos
    .groupBy(*CHAVES_PARCELA_AVALIACAO, "versao_parcela")
    .agg(F.max("valor_previsto_parcela").alias("valor_previsto_versao"))
)

parcelas_previstas_historicas = (
    valores_previstos_por_versao
    .groupBy(*CHAVES_PARCELA_AVALIACAO)
    .agg(
        F.sum("valor_previsto_versao").alias("valor_previsto_parcela"),
        F.countDistinct("versao_parcela").alias("qtd_versoes")
    )
)

eventos_pagamento_observados = (
    eventos_pagamento_historicos
    .filter(
        F.col("data_real_pagamento").isNotNull() &
        (F.col("data_real_pagamento") <= F.col("data_corte_features")) &
        F.col("valor_pago_parcela").isNotNull()
    )
    .groupBy(
        *CHAVES_PARCELA_AVALIACAO,
        "data_real_pagamento",
        "valor_pago_parcela"
    )
    .agg(F.countDistinct("versao_parcela").alias("qtd_versoes_evento"))
    .withColumnRenamed("valor_pago_parcela", "valor_pago_evento")
)

pagamentos_historicos_por_data = (
    eventos_pagamento_observados
    .groupBy(*CHAVES_PARCELA_AVALIACAO, "data_real_pagamento")
    .agg(F.sum("valor_pago_evento").alias("valor_pago_data"))
    .join(
        parcelas_previstas_historicas.select(*CHAVES_PARCELA_AVALIACAO, "valor_previsto_parcela"),
        CHAVES_PARCELA_AVALIACAO,
        "left"
    )
)

In [18]:
janela_pagamentos_historicos = (
    Window.partitionBy(*CHAVES_PARCELA_AVALIACAO)
    .orderBy("data_real_pagamento")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

pagamentos_historicos_acumulados = (
    pagamentos_historicos_por_data
    .withColumn("valor_pago_acumulado", F.sum("valor_pago_data").over(janela_pagamentos_historicos))
    .withColumn("saldo_residual_acumulado", F.col("valor_previsto_parcela") - F.col("valor_pago_acumulado"))
    .withColumn(
        "proporcao_saldo_residual_acumulado",
        F.when(
            F.col("valor_previsto_parcela") > 0,
            F.col("saldo_residual_acumulado") / F.col("valor_previsto_parcela")
        )
    )
    .withColumn(
        "data_quitacao_candidata",
        F.when(
            (F.col("valor_previsto_parcela") > 0) &
            (
                (F.col("valor_pago_acumulado") + F.lit(TOLERANCIA_PAGAMENTO) >= F.col("valor_previsto_parcela")) |
                (
                    (F.col("saldo_residual_acumulado") > 0) &
                    (F.col("saldo_residual_acumulado") <= F.lit(LIMITE_SALDO_RESIDUAL_ABSOLUTO)) &
                    (F.col("proporcao_saldo_residual_acumulado") <= F.lit(LIMITE_SALDO_RESIDUAL_PERCENTUAL))
                )
            ),
            F.col("data_real_pagamento")
        )
    )
)

resumo_pagamentos_historicos = (
    pagamentos_historicos_acumulados
    .groupBy(*CHAVES_PARCELA_AVALIACAO)
    .agg(
        F.sum("valor_pago_data").alias("valor_pago_observado_ate_corte"),
        F.min("data_quitacao_candidata").alias("data_quitacao_observada_ate_corte"),
        F.max("data_real_pagamento").alias("data_ultimo_pagamento_observado")
    )
)

In [19]:
parcelas_historicas_pagamento = (
    parcelas_previstas_historicas
    .join(resumo_pagamentos_historicos, CHAVES_PARCELA_AVALIACAO, "left")
    .fillna({"valor_pago_observado_ate_corte": 0.0})
    .withColumn(
        "saldo_residual_bruto",
        F.col("valor_previsto_parcela") - F.col("valor_pago_observado_ate_corte")
    )
    .withColumn(
        "proporcao_saldo_residual",
        F.when(
            F.col("valor_previsto_parcela") > 0,
            F.col("saldo_residual_bruto") / F.col("valor_previsto_parcela")
        )
    )
    .withColumn(
        "flag_saldo_residual_imaterial",
        (
            (F.col("saldo_residual_bruto") > 0) &
            (F.col("saldo_residual_bruto") <= F.lit(LIMITE_SALDO_RESIDUAL_ABSOLUTO)) &
            (F.col("proporcao_saldo_residual") <= F.lit(LIMITE_SALDO_RESIDUAL_PERCENTUAL))
        ).cast("int")
    )
    .withColumn(
        "data_quitacao_observada_ate_corte",
        F.when(
            (F.col("valor_previsto_parcela") > 0) &
            (
                (F.col("valor_pago_observado_ate_corte") + F.lit(TOLERANCIA_PAGAMENTO) >= F.col("valor_previsto_parcela")) |
                (F.col("flag_saldo_residual_imaterial") == 1)
            ),
            F.col("data_quitacao_observada_ate_corte")
        )
    )
    .withColumn(
        "status_pagamento_no_corte",
        F.when(
            F.col("valor_previsto_parcela").isNull() | (F.col("valor_previsto_parcela") <= 0),
            "VALOR_PREVISTO_INVALIDO"
        )
        .when(F.col("data_quitacao_observada_ate_corte").isNotNull(), "QUITADA")
        .when(F.col("valor_pago_observado_ate_corte") > F.lit(TOLERANCIA_PAGAMENTO), "PARCIAL")
        .otherwise("SEM_PAGAMENTO")
    )
    .withColumn(
        "dias_atraso",
        F.when(F.col("status_pagamento_no_corte") == "VALOR_PREVISTO_INVALIDO", F.lit(None).cast("int"))
        .when(
            F.col("data_quitacao_observada_ate_corte").isNotNull(),
            F.greatest(
                F.datediff("data_quitacao_observada_ate_corte", "data_prevista_pagamento"),
                F.lit(0)
            )
        )
        .otherwise(
            F.greatest(F.datediff("data_corte_features", "data_prevista_pagamento"), F.lit(0))
        )
    )
)

In [20]:
features_volume_frequencia_atrasos = (
    parcelas_historicas_pagamento
    .filter(F.col("dias_atraso").isNotNull())
    .groupBy("id_avaliacao")
    .agg(
        F.count("*").alias("qtd_parcelas_historicas"),
        F.sum((F.col("dias_atraso") > 0).cast("int")).alias("qtd_parcelas_com_atraso"),
        F.sum((F.col("dias_atraso") > 30).cast("int")).alias("qtd_parcelas_atraso_acima_30_dias"),
        F.sum((F.col("dias_atraso") > 60).cast("int")).alias("qtd_parcelas_atraso_acima_60_dias"),
        F.sum((F.col("dias_atraso") > 90).cast("int")).alias("qtd_parcelas_atraso_acima_90_dias")
    )
    .withColumn(
        "proporcao_parcelas_com_atraso",
        F.round(F.col("qtd_parcelas_com_atraso") / F.col("qtd_parcelas_historicas"), 6)
    )
    .withColumn("flag_historico_com_atraso", (F.col("qtd_parcelas_com_atraso") > 0).cast("int"))
    .withColumn(
        "flag_historico_atraso_acima_30_dias",
        (F.col("qtd_parcelas_atraso_acima_30_dias") > 0).cast("int")
    )
    .withColumn(
        "flag_historico_atraso_acima_60_dias",
        (F.col("qtd_parcelas_atraso_acima_60_dias") > 0).cast("int")
    )
    .withColumn(
        "flag_historico_atraso_acima_90_dias",
        (F.col("qtd_parcelas_atraso_acima_90_dias") > 0).cast("int")
    )
)

colunas_pagamento_zero = [
    "qtd_parcelas_historicas", "qtd_parcelas_com_atraso",
    "qtd_parcelas_atraso_acima_30_dias", "qtd_parcelas_atraso_acima_60_dias",
    "qtd_parcelas_atraso_acima_90_dias", "proporcao_parcelas_com_atraso",
    "flag_historico_com_atraso", "flag_historico_atraso_acima_30_dias",
    "flag_historico_atraso_acima_60_dias", "flag_historico_atraso_acima_90_dias"
]

features_historicas_pagamento = (
    bases_avaliacao.select("id_avaliacao", "tipo_base")
    .join(features_volume_frequencia_atrasos, "id_avaliacao", "left")
    .withColumn(
        "flag_sem_historico_pagamento",
        F.col("qtd_parcelas_historicas").isNull().cast("int")
    )
    .fillna(0, subset=colunas_pagamento_zero)
)

In [21]:
validacao_features_pagamento = (
    features_historicas_pagamento
    .groupBy("tipo_base")
    .agg(
        F.count("*").alias("qtd_avaliacoes"),
        F.countDistinct("id_avaliacao").alias("qtd_avaliacoes_distintas"),
        F.round(F.avg("flag_sem_historico_pagamento") * 100, 4).alias("perc_sem_historico_pagamento"),
        F.percentile_approx("qtd_parcelas_historicas", 0.5).alias("mediana_qtd_parcelas"),
        F.round(F.avg("flag_historico_com_atraso") * 100, 4).alias("perc_com_algum_atraso"),
        F.round(F.avg("flag_historico_atraso_acima_30_dias") * 100, 4).alias("perc_com_atraso_acima_30"),
        F.round(F.avg("flag_historico_atraso_acima_60_dias") * 100, 4).alias("perc_com_atraso_acima_60"),
        F.round(F.avg("flag_historico_atraso_acima_90_dias") * 100, 4).alias("perc_com_atraso_acima_90")
    )
    .orderBy("tipo_base")
)
visualizar(validacao_features_pagamento)

+---------------+--------------+------------------------+----------------------------+--------------------+---------------------+------------------------+------------------------+------------------------+
|tipo_base      |qtd_avaliacoes|qtd_avaliacoes_distintas|perc_sem_historico_pagamento|mediana_qtd_parcelas|perc_com_algum_atraso|perc_com_atraso_acima_30|perc_com_atraso_acima_60|perc_com_atraso_acima_90|
+---------------+--------------+------------------------+----------------------------+--------------------+---------------------+------------------------+------------------------+------------------------+
|DESENVOLVIMENTO|107419        |107419                  |35.6455                     |8                   |29.6065              |2.7397                  |1.1553                  |0.9384                  |
|SCORING        |40000         |40000                   |5.6525                      |21                  |48.235               |4.7825                  |2.2975                  |1

**Decisão**

As features de pagamento representam volume, recorrência e exposição a atrasos relevantes. Percentis e máximos de atraso não foram mantidos porque se mostraram concentrados em zero para a maior parte das avaliações e sensíveis a extremos. A ausência de histórico recebe flag própria e zeros apenas nas contagens, proporção e indicadores associados.

## 7. Consolidação e validação da base analítica

In [22]:
def selecionar_features_sem_chaves_redundantes(dataframe):
    return dataframe.select(
        "id_avaliacao",
        *[c for c in dataframe.columns if c not in {"id_avaliacao", "tipo_base"}]
    )

base_analitica_features = (
    features_cadastrais
    .join(selecionar_features_sem_chaves_redundantes(features_historicas_credito_status), "id_avaliacao", "left")
    .join(selecionar_features_sem_chaves_redundantes(features_historicas_credito_valores), "id_avaliacao", "left")
    .join(selecionar_features_sem_chaves_redundantes(features_historicas_pagamento), "id_avaliacao", "left")
)

validacao_integridade_base = (
    base_analitica_features
    .groupBy("tipo_base")
    .agg(
        F.count("*").alias("qtd_registros"),
        F.countDistinct("id_avaliacao").alias("qtd_avaliacoes_distintas"),
        F.sum(F.col("id_avaliacao").isNull().cast("int")).alias("qtd_ids_nulos")
    )
    .withColumn("qtd_duplicados", F.col("qtd_registros") - F.col("qtd_avaliacoes_distintas"))
    .withColumn("qtd_colunas", F.lit(len(base_analitica_features.columns)))
    .orderBy("tipo_base")
)
visualizar(validacao_integridade_base)

+---------------+-------------+------------------------+-------------+--------------+-----------+
|tipo_base      |qtd_registros|qtd_avaliacoes_distintas|qtd_ids_nulos|qtd_duplicados|qtd_colunas|
+---------------+-------------+------------------------+-------------+--------------+-----------+
|DESENVOLVIMENTO|107419       |107419                  |0            |0             |92         |
|SCORING        |40000        |40000                   |0            |0             |92         |
+---------------+-------------+------------------------+-------------+--------------+-----------+



In [23]:
tipos_colunas = dict(base_analitica_features.dtypes)
colunas_features = [
    coluna for coluna in base_analitica_features.columns
    if coluna not in {"id_avaliacao", "tipo_base"}
]

def condicao_nulo(coluna):
    valor = F.col(coluna)
    return valor.isNull() | (
        F.isnan(valor) if tipos_colunas[coluna] in {"double", "float"} else F.lit(False)
    )

def valor_por_base(base, coluna, nome):
    return F.max(F.when(F.col("tipo_base") == base, F.col(coluna))).alias(nome)

agregacoes_nulos = [
    F.sum(F.when(condicao_nulo(coluna), 1).otherwise(0)).alias(coluna)
    for coluna in colunas_features
]

estrutura_nulos = [
    F.struct(F.lit(coluna).alias("variavel"), F.col(coluna).alias("qtd_nulos"))
    for coluna in colunas_features
]

resumo_nulos = (
    base_analitica_features
    .groupBy("tipo_base")
    .agg(F.count("*").alias("qtd_avaliacoes"), *agregacoes_nulos)
)

validacao_nulos_completa = (
    resumo_nulos
    .select(
        "tipo_base", "qtd_avaliacoes",
        F.explode(F.array(*estrutura_nulos)).alias("resultado")
    )
    .select("tipo_base", "qtd_avaliacoes", "resultado.*")
    .withColumn("qtd_validos", F.col("qtd_avaliacoes") - F.col("qtd_nulos"))
    .withColumn(
        "perc_nulos",
        F.round(F.col("qtd_nulos") / F.col("qtd_avaliacoes") * 100, 4)
    )
)

comparativo_nulos = (
    validacao_nulos_completa
    .groupBy("variavel")
    .agg(
        valor_por_base("DESENVOLVIMENTO", "qtd_validos", "qtd_validos_desenvolvimento"),
        valor_por_base("DESENVOLVIMENTO", "perc_nulos", "perc_nulos_desenvolvimento"),
        valor_por_base("SCORING", "qtd_validos", "qtd_validos_scoring"),
        valor_por_base("SCORING", "perc_nulos", "perc_nulos_scoring")
    )
    .withColumn(
        "diferenca_abs_pp",
        F.round(
            F.abs(F.col("perc_nulos_scoring") - F.col("perc_nulos_desenvolvimento")),
            4
        )
    )
    .filter(
        (F.col("perc_nulos_desenvolvimento") >= 1)
        | (F.col("perc_nulos_scoring") >= 1)
    )
    .orderBy(F.desc("perc_nulos_desenvolvimento"))
)

visualizar(comparativo_nulos, n=100)

+---------------------------------------------+---------------------------+--------------------------+-------------------+------------------+----------------+
|variavel                                     |qtd_validos_desenvolvimento|perc_nulos_desenvolvimento|qtd_validos_scoring|perc_nulos_scoring|diferenca_abs_pp|
+---------------------------------------------+---------------------------+--------------------------+-------------------+------------------+----------------+
|media_qtd_parcelas_planejadas_anterior       |5                          |99.9953                   |1                  |99.9975           |0.0022          |
|media_razao_liberado_solicitado              |965                        |99.1016                   |3263               |91.8425           |7.2591          |
|perc_credito_liberado_menor_solicitado       |965                        |99.1016                   |3263               |91.8425           |7.2591          |
|soma_valor_liberado_contratos_ativos_no_corte

In [24]:
COLUNAS_EXCLUIR_BAIXA_COBERTURA = [
    "media_qtd_parcelas_planejadas_anterior",
    "media_razao_liberado_solicitado",
    "perc_credito_liberado_menor_solicitado"
]

base_analitica_pre_modelagem = base_analitica_features.drop(*COLUNAS_EXCLUIR_BAIXA_COBERTURA)

colunas_identificacao = [
    c for c in ["id_avaliacao", "tipo_base", "id_cliente", "id_contrato", "id_contrato_avaliado"]
    if c in base_analitica_pre_modelagem.columns
]
colunas_data = [
    c for c, tipo in base_analitica_pre_modelagem.dtypes
    if tipo in {"date", "timestamp"}
]
colunas_controle_qualidade = [
    c for c in ["qtd_status_classificados", "flag_inconsistencia_soma_status"]
    if c in base_analitica_pre_modelagem.columns
]
colunas_auditoria = [
    c for c in COLUNAS_OPERACAO_SOMENTE_AUDITORIA
    if c in base_analitica_pre_modelagem.columns
]

colunas_governanca = [
    c for c in COLUNAS_CADASTRAIS_SOMENTE_AUDITORIA
    if c in base_analitica_pre_modelagem.columns
]

colunas_nao_modelaveis = list(dict.fromkeys(
    colunas_identificacao + colunas_data + colunas_controle_qualidade
    + colunas_auditoria + colunas_governanca
))
colunas_candidatas_modelagem = [
    c for c in base_analitica_pre_modelagem.columns
    if c not in colunas_nao_modelaveis
]

tipos_pre_modelagem = dict(base_analitica_pre_modelagem.dtypes)
tipos_numericos = {"tinyint", "smallint", "int", "bigint", "float", "double"}
colunas_numericas = [
    c for c in colunas_candidatas_modelagem
    if tipos_pre_modelagem[c] in tipos_numericos or tipos_pre_modelagem[c].startswith("decimal")
]

print(f"Colunas antes da exclusão por cobertura: {len(base_analitica_features.columns)}")
print(f"Colunas após a exclusão por cobertura: {len(base_analitica_pre_modelagem.columns)}")
print(f"Features candidatas: {len(colunas_candidatas_modelagem)}")
print(f"Features numéricas: {len(colunas_numericas)}")

Colunas antes da exclusão por cobertura: 92
Colunas após a exclusão por cobertura: 89
Features candidatas: 72
Features numéricas: 61


In [25]:
base_desenvolvimento_pre_modelagem = base_analitica_pre_modelagem.filter(
    F.col("tipo_base") == "DESENVOLVIMENTO"
)

resumo_variabilidade = base_desenvolvimento_pre_modelagem.agg(*[
    F.countDistinct(F.col(c)).alias(c)
    for c in colunas_candidatas_modelagem
])

validacao_sem_variabilidade = (
    resumo_variabilidade
    .select(F.explode(F.array(*[
        F.struct(
            F.lit(c).alias("variavel"),
            F.lit(tipos_pre_modelagem[c]).alias("tipo_dado"),
            F.col(c).alias("qtd_valores_distintos")
        )
        for c in colunas_candidatas_modelagem
    ])).alias("resultado"))
    .select("resultado.*")
    .filter(F.col("qtd_valores_distintos") <= 1)
    .orderBy("qtd_valores_distintos", "variavel")
)

resumo_infinitos = (
    base_analitica_pre_modelagem
    .groupBy("tipo_base")
    .agg(*[
        F.sum(F.col(c).isin(float("inf"), float("-inf")).cast("int")).alias(c)
        for c in colunas_numericas
    ])
)

validacao_infinitos = (
    resumo_infinitos
    .select(
        "tipo_base",
        F.explode(F.array(*[
            F.struct(F.lit(c).alias("variavel"), F.col(c).alias("qtd_valores_infinitos"))
            for c in colunas_numericas
        ])).alias("resultado")
    )
    .select("tipo_base", "resultado.*")
    .filter(F.col("qtd_valores_infinitos") > 0)
)

visualizar(validacao_sem_variabilidade)
visualizar(validacao_infinitos)

+--------+---------+---------------------+
|variavel|tipo_dado|qtd_valores_distintos|
+--------+---------+---------------------+
+--------+---------+---------------------+

+---------+--------+---------------------+
|tipo_base|variavel|qtd_valores_infinitos|
+---------+--------+---------------------+
+---------+--------+---------------------+



**Decisões de adequação**

- Três variáveis foram excluídas por cobertura praticamente inexistente no desenvolvimento.
- Identificadores, datas, controles de qualidade e colunas de auditoria não entram diretamente no modelo.
- Ausências estruturais de histórico permanecem acompanhadas por flags; imputações dependentes da distribuição serão aprendidas somente no treino.
- Seleção final, encoding e imputação não usam estatísticas da base de scoring.

## 8. Separação e salvamento das bases intermediárias

Desenvolvimento e scoring conservam o mesmo contrato de features. O target será integrado explicitamente no Notebook 04.

In [26]:
colunas_rastreabilidade = [
    c for c in ["id_avaliacao", "tipo_base", "id_cliente", "id_contrato", "data_avaliacao", "data_corte_features"]
    if c in base_analitica_pre_modelagem.columns
]
colunas_base_modelagem = colunas_rastreabilidade + colunas_candidatas_modelagem

base_desenvolvimento_features = (
    base_analitica_pre_modelagem
    .filter(F.col("tipo_base") == "DESENVOLVIMENTO")
    .select(*colunas_base_modelagem)
)

base_scoring_features = (
    base_analitica_pre_modelagem
    .filter(F.col("tipo_base") == "SCORING")
    .select(*colunas_base_modelagem)
)

validacao_bases_finais = (
    base_desenvolvimento_features
    .agg(
        F.lit("DESENVOLVIMENTO").alias("tipo_base"),
        F.count("*").alias("qtd_registros"),
        F.countDistinct("id_avaliacao").alias("qtd_avaliacoes_distintas"),
        F.sum(F.col("id_avaliacao").isNull().cast("int")).alias("qtd_ids_avaliacao_nulos"),
        F.sum(F.col("id_cliente").isNull().cast("int")).alias("qtd_ids_cliente_nulos")
    )
    .unionByName(
        base_scoring_features.agg(
            F.lit("SCORING").alias("tipo_base"),
            F.count("*").alias("qtd_registros"),
            F.countDistinct("id_avaliacao").alias("qtd_avaliacoes_distintas"),
            F.sum(F.col("id_avaliacao").isNull().cast("int")).alias("qtd_ids_avaliacao_nulos"),
            F.sum(F.col("id_cliente").isNull().cast("int")).alias("qtd_ids_cliente_nulos")
        )
    )
    .withColumn("qtd_features_candidatas", F.lit(len(colunas_candidatas_modelagem)))
    .withColumn("qtd_colunas_base", F.lit(len(colunas_base_modelagem)))
)

visualizar(validacao_bases_finais)
print("Schemas compatíveis:", base_desenvolvimento_features.dtypes == base_scoring_features.dtypes)

+---------------+-------------+------------------------+-----------------------+---------------------+-----------------------+----------------+
|tipo_base      |qtd_registros|qtd_avaliacoes_distintas|qtd_ids_avaliacao_nulos|qtd_ids_cliente_nulos|qtd_features_candidatas|qtd_colunas_base|
+---------------+-------------+------------------------+-----------------------+---------------------+-----------------------+----------------+
|DESENVOLVIMENTO|107419       |107419                  |0                      |0                    |72                     |78              |
|SCORING        |40000        |40000                   |0                      |0                    |72                     |78              |
+---------------+-------------+------------------------+-----------------------+---------------------+-----------------------+----------------+

Schemas compatíveis: True


In [27]:
def salvar_parquet_local(dataframe_spark, caminho):
    caminho = Path(caminho)
    caminho.parent.mkdir(parents=True, exist_ok=True)

    dataframe_spark.toPandas().to_parquet(
        caminho,
        index=False,
        engine="pyarrow"
    )


CAMINHO_BASE_DESENVOLVIMENTO = juntar_caminho(
    OUTPUT_PATH,
    "base_desenvolvimento_features.parquet"
)

CAMINHO_BASE_SCORING = juntar_caminho(
    OUTPUT_PATH,
    "base_scoring_features.parquet"
)

salvar_parquet_local(
    base_desenvolvimento_features,
    CAMINHO_BASE_DESENVOLVIMENTO
)

salvar_parquet_local(
    base_scoring_features,
    CAMINHO_BASE_SCORING
)

base_desenvolvimento_validacao = spark.read.parquet(
    CAMINHO_BASE_DESENVOLVIMENTO
)

base_scoring_validacao = spark.read.parquet(
    CAMINHO_BASE_SCORING
)

print(
    "Desenvolvimento:",
    base_desenvolvimento_validacao.count(),
    len(base_desenvolvimento_validacao.columns)
)

print(
    "Scoring:",
    base_scoring_validacao.count(),
    len(base_scoring_validacao.columns)
)

schema_features_desenvolvimento = dict(
    base_desenvolvimento_validacao
    .select(*colunas_candidatas_modelagem)
    .dtypes
)

schema_features_scoring = dict(
    base_scoring_validacao
    .select(*colunas_candidatas_modelagem)
    .dtypes
)

diferencas_schema = {
    coluna: (
        schema_features_desenvolvimento.get(coluna),
        schema_features_scoring.get(coluna)
    )
    for coluna in colunas_candidatas_modelagem
    if (
        schema_features_desenvolvimento.get(coluna)
        != schema_features_scoring.get(coluna)
    )
}

if diferencas_schema:
    raise ValueError(
        f"Features com tipos incompatíveis: {diferencas_schema}"
    )

print("Contrato de features persistido compatível: True")

Desenvolvimento: 107419 78
Scoring: 40000 78
Contrato de features persistido compatível: True


## 9. Conclusões

- A base analítica foi construída em uma linha por avaliação, sem perda de desenvolvimento ou scoring.
- Todas as features históricas respeitam o corte temporal da avaliação.
- A lógica de pagamentos utiliza quitação acumulada e preserva parcelas parciais como abertas até o corte.
- Variáveis com cobertura insuficiente, sem capacidade de aprendizado ou destinadas apenas à auditoria foram separadas do conjunto preditivo.
- Desenvolvimento e scoring foram persistidos com o mesmo contrato de features, sem que a população de scoring participasse do aprendizado.
- Diferenças de cobertura histórica entre as populações permanecem como risco explícito de estabilidade.
- O target é observado somente para contratos aprovados e performados; portanto, há viés de aprovação. Inferência de rejeitados é registrada como evolução, sem criar rótulos artificiais.



## 10. Próximos passos

O Notebook 04 deverá:

1. integrar o target final à base de desenvolvimento;
2. definir divisão temporal de treino, validação e teste;
3. aprender imputação, encoding e seleção apenas no treino;
4. comparar modelos simples e justificáveis;
5. avaliar discriminação, estabilidade e calibração;
6. propor política de crédito com faixas de risco e impacto de aprovação;
7. gerar `submissao_case.csv`.